# 2a. Provider catalog, policies and connectors

The Provider publishes what it offers as a DCAT catalog: datasets, their distributions (how the
data is delivered) and ODRL policies (the terms under which it may be used). A connector instance
bound to each distribution tells the Provider how to reach the real backend when a transfer
starts; in this notebook the connectors use no authentication (`NO_AUTH`).

Steps:
1. Onboarding (`setup_auth.ipynb`), guarded.
2. Resolve the Provider's main catalog and data service (`/api/v1/catalog-agent/catalogs/main`, `/data-services/main`).
3. Create a dataset with a PULL distribution (`http+pull`) and a PUSH distribution (`http+push`) (`/api/v1/catalog-agent/datasets`, `/distributions`).
4. Attach an ODRL offer instantiated from a policy template (`/api/v1/catalog-agent/policy-templates`, `/instantiate-odrl-offer`).
5. Attach two ODRL offers written by hand and list the dataset's offers (`/api/v1/catalog-agent/odrl-policies`, `/odrl-policies/entity/{id}`).
6. Register a PULL connector template, instantiate it on the PULL distribution and inspect it (`/api/v1/connector/templates`, `/instances`, `/instances/distribution/{id}`).
7. Register a PUSH connector template, instantiate it on the PUSH distribution and inspect it.
8. The Consumer discovers the catalog and dataset over DSP (`/dsp/current/catalog/rpc/setup-catalog-request`, `setup-dataset-request`) and inspects the cached peer catalog (`/api/v1/catalog-agent/peer-catalogs/{peerId}`).

**Runs on its own.** It leaves `target_id`, `policy_id` and `data_space_provider_dsp_endpoint`
ready for the negotiation notebooks, and sets `catalog_setup = "no_auth"` so that the transfer
notebooks know which variant is loaded. Each run creates a new dataset. `setup_catalog_policies_oauth.ipynb` is the
same flow with connectors that authenticate with a bearer token.

## 1. Prerequisites

The guard runs the onboarding notebook only when this kernel has not run it yet. `%run` on a
notebook executes its cells in this same namespace, so the URLs, DIDs and `show` it defines are
available below.

In [1]:
if not globals().get("onboarded"):
    %run ./setup_auth.ipynb

Authority URL: http://127.0.0.1:1500
Consumer URL:  http://127.0.0.1:1100
Provider URL:  http://127.0.0.1:1200
Wallet linked for Authority (http://127.0.0.1:1500): wallet id=fafnir-local, name=fafnir-wallet
Wallet linked for Consumer (http://127.0.0.1:1100): wallet id=fafnir-local, name=fafnir-wallet
Wallet linked for Provider (http://127.0.0.1:1200): wallet id=fafnir-local, name=fafnir-wallet
HTTP 200 GET http://127.0.0.1:1100/api/v1/mates/myself
{
  "participant_id": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ",
  "participant_n

## 2. Main catalog and data service (`/api/v1/catalog-agent/*`)

Every Provider starts with a `main` catalog and a `main` data service. The data service is the
Provider's DSP endpoint; distributions point to it through `dcatAccessService`.

In [2]:
catalog = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/catalogs/main", headers=json_header))
catalog_id = catalog["id"]

HTTP 200 GET http://127.0.0.1:1200/api/v1/catalog-agent/catalogs/main
{
  "dctConformsTo": null,
  "dctCreator": null,
  "dctIdentifier": "urn:catalog:7d98ad53-dc19-4f07-aa14-e91292d25047",
  "dctIssued": "2026-10-16T21:16:00.204583Z",
  "dctModified": null,
  "dctTitle": null,
  "dspaceMainCatalog": true,
  "dspaceParticipantId": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
  "foafHomePage": null,
  "id": "urn:catalog:7d98ad53-dc19-4f07-aa14-e91292d25047",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Public"
}


In [3]:
data_service = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/data-services/main", headers=json_header))
data_service_id = data_service["id"]

HTTP 200 GET http://127.0.0.1:1200/api/v1/catalog-agent/data-services/main
{
  "catalogId": "urn:catalog:7d98ad53-dc19-4f07-aa14-e91292d25047",
  "dcatEndpointDescription": null,
  "dcatEndpointUrl": "http://127.0.0.1:1200/dsp/current",
  "dctConformsTo": null,
  "dctCreator": null,
  "dctDescription": null,
  "dctIdentifier": "urn:data-service:202eb56d-a8e6-4cee-b49f-fa4e2b8e1c5c",
  "dctIssued": "2026-10-16T21:16:00.278647Z",
  "dctModified": null,
  "dctTitle": null,
  "dspaceMainDataService": true,
  "id": "urn:data-service:202eb56d-a8e6-4cee-b49f-fa4e2b8e1c5c",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Public"
}


## 3. Dataset and distributions (`/api/v1/catalog-agent/datasets`, `/distributions`)

A dataset is the unit of negotiation: a contract agreement always targets a dataset. Its
distributions are the ways to obtain it. The format of a distribution is the `format` a transfer
request asks for later:
- `http+pull`: the Consumer pulls the data from an endpoint the Provider hands over.
- `http+push`: the Provider pushes the data to a sink the Consumer names.

In [4]:
payload = {
    "dctTitle": "My dataspace dataset",
    "catalogId": catalog_id,
}
dataset = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/datasets", headers=json_header, json=payload))
dataset_id = dataset["id"]

HTTP 200 POST http://127.0.0.1:1200/api/v1/catalog-agent/datasets
{
  "catalogId": "urn:catalog:7d98ad53-dc19-4f07-aa14-e91292d25047",
  "dctConformsTo": null,
  "dctCreator": null,
  "dctDescription": null,
  "dctIdentifier": "urn:dataset:9b38dfda-bcb8-4968-a826-294c7905c929",
  "dctIssued": "2026-10-17T14:00:46.335738Z",
  "dctModified": null,
  "dctTitle": "My dataspace dataset",
  "id": "urn:dataset:9b38dfda-bcb8-4968-a826-294c7905c929",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Private"
}


In [5]:
payload = {
    "dctTitle": "Distribution of my dataset",
    "dctDescription": "Pull access to my dataset",
    "dcatAccessService": data_service_id,
    "datasetId": dataset_id,
    "dctFormats": "http+pull",
}
distribution = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/distributions", headers=json_header, json=payload))
distribution_pull_id = distribution["id"]

HTTP 200 POST http://127.0.0.1:1200/api/v1/catalog-agent/distributions
{
  "datasetId": "urn:dataset:9b38dfda-bcb8-4968-a826-294c7905c929",
  "dcatAccessService": "urn:data-service:202eb56d-a8e6-4cee-b49f-fa4e2b8e1c5c",
  "dctDescription": "Pull access to my dataset",
  "dctFormat": "http+pull",
  "dctIssued": "2026-10-17T14:00:46.380228Z",
  "dctModified": null,
  "dctTitle": "Distribution of my dataset",
  "id": "urn:distribution:c554984d-6de9-46db-9189-5e117a78faf7",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Private"
}


In [6]:
payload = {
    "dctTitle": "Distribution of a streaming dataset",
    "dctDescription": "Push delivery of my streaming dataset",
    "dcatAccessService": data_service_id,
    "datasetId": dataset_id,
    "dctFormats": "http+push",
}
distribution = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/distributions", headers=json_header, json=payload))
distribution_push_id = distribution["id"]

dataset_distributions = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/distributions/dataset/{dataset_id}", headers=json_header))
print(f"Distributions on dataset {dataset_id}: {[d['dctFormat'] for d in dataset_distributions]}")

HTTP 200 POST http://127.0.0.1:1200/api/v1/catalog-agent/distributions
{
  "datasetId": "urn:dataset:9b38dfda-bcb8-4968-a826-294c7905c929",
  "dcatAccessService": "urn:data-service:202eb56d-a8e6-4cee-b49f-fa4e2b8e1c5c",
  "dctDescription": "Push delivery of my streaming dataset",
  "dctFormat": "http+push",
  "dctIssued": "2026-10-17T14:00:46.420753Z",
  "dctModified": null,
  "dctTitle": "Distribution of a streaming dataset",
  "id": "urn:distribution:b9251cb8-a2f9-4772-9330-5bd4fe7444b9",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Private"
}
HTTP 200 GET http://127.0.0.1:1200/api/v1/catalog-agent/distributions/dataset/urn:dataset:9b38dfda-bcb8-4968-a826-294c7905c929
[
  {
    "datasetId": "urn:dataset:9b38dfda-bcb8-4968-a826-294c7905c929",
    "dcatAccessService": "urn:data-service:202eb56d-a8e6-4cee-b49f-fa4e2b8e1c5c",
    "dctDescription": "Pull access to my dataset",
    "dctFormat": "http+pull",
    "dctIssued": "2026-10-17T14:00:46.380228Z",
    "dctModified"

## 4. ODRL offer from a policy template (`/api/v1/catalog-agent/policy-templates`)

Policy templates are parameterised ODRL policies. The agent seeds `policy-1`, whose single
permission constrains `bla` to the value of the `$bla` parameter. Instantiating it attaches the
resulting offer to the dataset.

In [7]:
templates = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/policy-templates", headers=json_header))

HTTP 200 GET http://127.0.0.1:1200/api/v1/catalog-agent/policy-templates
{
  "items": [
    {
      "author": "",
      "content": {
        "permission": [
          {
            "action": "use",
            "constraint": [
              {
                "leftOperand": "bla",
                "operator": "eq",
                "rightOperand": "$bla"
              }
            ]
          }
        ]
      },
      "date": "2026-10-16T21:16:00.293532Z",
      "description": "Mi example super policy",
      "id": "policy-1",
      "parameters": {
        "$bla": {
          "dataType": "string",
          "restrictions": {},
          "ui": {
            "hidden": false
          }
        }
      },
      "title": "Mi example policy",
      "userId": "system",
      "userRole": "/admin",
      "version": "1.0",
      "visibility": "Public"
    }
  ],
  "total": 1
}


In [8]:
payload = {
    "id": "policy-1",
    "version": "1.0",
    "parameters": {
        "$bla": "did:web:university.upm.es"
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Research access for UPM: grants use to did:web:university.upm.es.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/policy-templates/instantiate-odrl-offer", headers=json_header, json=payload))

HTTP 202 POST http://127.0.0.1:1200/api/v1/catalog-agent/policy-templates/instantiate-odrl-offer
{
  "createdAt": "2026-10-17T14:00:46.491649Z",
  "description": "Research access for UPM: grants use to did:web:university.upm.es.",
  "entity": "urn:dataset:9b38dfda-bcb8-4968-a826-294c7905c929",
  "entityType": "Dataset",
  "id": "urn:odrl-policy:ac97d7e3-89bb-4099-ba4b-01db3e76e7d9",
  "instantiationParameters": {
    "$bla": "did:web:university.upm.es"
  },
  "odrlOffer": {
    "permission": [
      {
        "action": "use",
        "constraint": [
          {
            "leftOperand": "bla",
            "operator": "eq",
            "rightOperand": "did:web:university.upm.es"
          }
        ]
      }
    ]
  },
  "sourceTemplateId": "policy-1",
  "sourceTemplateVersion": "1.0",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Private"
}


## 5. ODRL offers written by hand (`/api/v1/catalog-agent/odrl-policies`)

An offer can also be posted as plain ODRL: permissions, obligations (duties) and prohibitions,
each with optional constraints. `GET /api/v1/catalog-agent/odrl-policies/entity/{dataset_id}` lists
all offers attached to the dataset.

In [9]:
payload = {
    "odrlOffer": {
        "permission": [{
            "action": "use",
            "constraint": [
                {"leftOperand": "dateTime", "operator": "gteq", "rightOperand": "2026-01-01T00:00:00Z"},
                {"leftOperand": "dateTime", "operator": "lteq", "rightOperand": "2026-12-31T23:59:59Z"},
                {"leftOperand": "purpose", "operator": "isA", "rightOperand": "commercial"},
            ],
        }],
        "obligation": [{
            "action": "attribute",
            "constraint": [{"leftOperand": "recipient", "operator": "isA", "rightOperand": "public"}],
        }],
        "prohibition": [{"action": "derive", "constraint": []}],
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Commercial licence for 2026: commercial use with mandatory attribution; derivation is prohibited.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/odrl-policies", headers=json_header, json=payload))

HTTP 200 POST http://127.0.0.1:1200/api/v1/catalog-agent/odrl-policies
{
  "createdAt": "2026-10-17T14:00:46.682886Z",
  "description": "Commercial licence for 2026: commercial use with mandatory attribution; derivation is prohibited.",
  "entity": "urn:dataset:9b38dfda-bcb8-4968-a826-294c7905c929",
  "entityType": "Dataset",
  "id": "urn:odrl-policy:89f8e97f-1810-4a51-a5a6-148d85983556",
  "instantiationParameters": null,
  "odrlOffer": {
    "obligation": [
      {
        "action": "attribute",
        "constraint": [
          {
            "leftOperand": "recipient",
            "operator": "isA",
            "rightOperand": "public"
          }
        ]
      }
    ],
    "permission": [
      {
        "action": "use",
        "constraint": [
          {
            "leftOperand": "dateTime",
            "operator": "gteq",
            "rightOperand": "2026-01-01T00:00:00Z"
          },
          {
            "leftOperand": "dateTime",
            "operator": "lteq",
         

In [10]:
payload = {
    "odrlOffer": {
        "permission": [{
            "action": "read",
            "constraint": [
                {"leftOperand": "dateTime", "operator": "gteq", "rightOperand": "2026-03-01T00:00:00Z"},
                {"leftOperand": "dateTime", "operator": "lteq", "rightOperand": "2026-09-01T00:00:00Z"},
                {"leftOperand": "purpose", "operator": "isA", "rightOperand": "research"},
                {"leftOperand": "count", "operator": "lteq", "rightOperand": "10000"},
            ],
        }],
        "obligation": [],
        "prohibition": [
            {"action": "distribute", "constraint": []},
            {"action": "reproduce", "constraint": []},
        ],
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Research trial (Mar-Sep 2026): read only, up to 10 000 requests, research purposes; no redistribution or reproduction.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/odrl-policies", headers=json_header, json=payload))

dataset_policies = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/odrl-policies/entity/{dataset_id}", headers=json_header))
print(f"Total policies on dataset {dataset_id}: {len(dataset_policies)}")

HTTP 200 POST http://127.0.0.1:1200/api/v1/catalog-agent/odrl-policies
{
  "createdAt": "2026-10-17T14:00:46.832964Z",
  "description": "Research trial (Mar-Sep 2026): read only, up to 10 000 requests, research purposes; no redistribution or reproduction.",
  "entity": "urn:dataset:9b38dfda-bcb8-4968-a826-294c7905c929",
  "entityType": "Dataset",
  "id": "urn:odrl-policy:d7d6e382-bc19-4a91-ac22-1d1f3d7acb36",
  "instantiationParameters": null,
  "odrlOffer": {
    "obligation": [],
    "permission": [
      {
        "action": "read",
        "constraint": [
          {
            "leftOperand": "dateTime",
            "operator": "gteq",
            "rightOperand": "2026-03-01T00:00:00Z"
          },
          {
            "leftOperand": "dateTime",
            "operator": "lteq",
            "rightOperand": "2026-09-01T00:00:00Z"
          },
          {
            "leftOperand": "purpose",
            "operator": "isA",
            "rightOperand": "research"
          },
        

## 6. PULL connector (`/api/v1/connector/*`)

A connector template describes how to reach a backend, with `{{__NAME__}}` placeholders; an
instance fills the parameters and binds the template to a distribution. In PULL mode the template
describes the data access the Consumer gets once the transfer starts.
`GET /api/v1/connector/instances/distribution/{id}` retrieves the instance bound to a distribution.

In [11]:
payload = {
    "authentication": {"type": "NO_AUTH"},
    "interaction": {
        "mode": "PULL",
        "dataAccess": {
            "protocol": "HTTP",
            "urlTemplate": "{{__ACCESS_URL__}}",
            "method": "{{__ACCESS_METHODS__}}",
            "headers": "{{__ACCESS_HEADERS__}}"
        }
    },
    "parameters": [
        {"paramType": "STRING", "name": "ACCESS_URL", "title": "Url", "required": True},
        {"paramType": "VEC<STRING>", "name": "ACCESS_METHODS", "title": "Methods", "required": True},
        {"paramType": "MAP<STRING,STRING>", "name": "ACCESS_HEADERS", "title": "Headers", "required": True}
    ]
}
template = show(requests.post(f"{data_space_provider}/api/v1/connector/templates", headers=json_header, json=payload))

HTTP 200 POST http://127.0.0.1:1200/api/v1/connector/templates
{
  "name": "urn:connector-template:9cdf1e03-5fc7-469f-90c2-bbb8c556b5ab",
  "author": "admin",
  "description": null,
  "version": "1.0",
  "createdAt": "2026-10-17T14:00:47.143639Z",
  "authentication": {
    "type": "NO_AUTH"
  },
  "interaction": {
    "mode": "PULL",
    "dataAccess": {
      "protocol": "HTTP",
      "urlTemplate": "{{__ACCESS_URL__}}",
      "method": "{{__ACCESS_METHODS__}}",
      "headers": "{{__ACCESS_HEADERS__}}",
      "bodyTemplate": null
    }
  },
  "parameters": [
    {
      "name": "ACCESS_URL",
      "title": "Url",
      "description": null,
      "paramType": "STRING",
      "required": true,
      "defaultValue": null
    },
    {
      "name": "ACCESS_METHODS",
      "title": "Methods",
      "description": null,
      "paramType": "VEC<STRING>",
      "required": true,
      "defaultValue": null
    },
    {
      "name": "ACCESS_HEADERS",
      "title": "Headers",
      "descriptio

In [12]:
payload = {
    "templateName": template["name"],
    "templateVersion": template["version"],
    "distributionId": distribution_pull_id,
    "metadata": {
        "description": "Pull access to the demo backend",
        "ownerId": "tutorial"
    },
    "dryRun": False,
    "parameters": {
        "ACCESS_URL": "http://127.0.0.1:8081",
        "ACCESS_METHODS": ["GET", "HEAD"],
        "ACCESS_HEADERS": {"Content-type": "application/json"}
    }
}
instance = show(requests.post(f"{data_space_provider}/api/v1/connector/instances", headers=json_header, json=payload))
pull_connector = show(requests.get(f"{data_space_provider}/api/v1/connector/instances/distribution/{distribution_pull_id}", headers=json_header))

HTTP 200 POST http://127.0.0.1:1200/api/v1/connector/instances
{
  "id": "urn:connector-instance:c05c3b5d-0872-4e4f-841f-e640b9df6811",
  "userId": "system",
  "name": "urn:connector-template:9cdf1e03-5fc7-469f-90c2-bbb8c556b5ab",
  "author": null,
  "description": null,
  "version": "1.0",
  "createdAt": "2026-10-17T14:00:47.317499Z",
  "authenticationConfig": {
    "type": "NO_AUTH"
  },
  "interaction": {
    "mode": "PULL",
    "dataAccess": {
      "protocol": "HTTP",
      "urlTemplate": "http://127.0.0.1:8081",
      "method": [
        "GET",
        "HEAD"
      ],
      "headers": {
        "Content-type": "application/json"
      },
      "bodyTemplate": null
    }
  },
  "distributionId": "urn:distribution:c554984d-6de9-46db-9189-5e117a78faf7"
}
HTTP 200 GET http://127.0.0.1:1200/api/v1/connector/instances/distribution/urn:distribution:c554984d-6de9-46db-9189-5e117a78faf7
{
  "id": "urn:connector-instance:c05c3b5d-0872-4e4f-841f-e640b9df6811",
  "userId": "system",
  "name"

## 7. PUSH connector (`/api/v1/connector/*`)

In PUSH mode the template says how to subscribe the Provider to the backend when a transfer
starts, and how to unsubscribe when it stops. `SUB_URL` and `UNSUB_URL` are filled by the
instance; the `SYS_*` and `RUNTIME_*` placeholders are filled by the agent at run time (its own
URL, the ingress of the transfer, and the subscription id the backend answered).

In [13]:
payload = {
    "authentication": {"type": "NO_AUTH"},
    "interaction": {
        "mode": "PUSH",
        "subscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__SUB_URL__}}",
            "method": "POST",
            "headers": {},
            "bodyTemplate": json.dumps({
                "url": "{{__SYS_OWN_URL_DOCKER__}}{{__RUNTIME_INGRESS__}}",
                "event_type": "hotel_waste_generated"
            })
        },
        "unsubscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__UNSUB_URL__}}/{{__RUNTIME_JSON_{.subscription.data.ID}__}}",
            "method": "POST",
            "headers": {},
            "bodyTemplate": json.dumps({})
        }
    },
    "parameters": [
        {"paramType": "STRING", "name": "SUB_URL", "title": "Subscription URL", "required": True},
        {"paramType": "STRING", "name": "UNSUB_URL", "title": "Unsubscription URL", "required": True}
    ]
}
template = show(requests.post(f"{data_space_provider}/api/v1/connector/templates", headers=json_header, json=payload))

HTTP 200 POST http://127.0.0.1:1200/api/v1/connector/templates
{
  "name": "urn:connector-template:cddb3021-091a-4e2a-94d7-9c1920abaa86",
  "author": "admin",
  "description": null,
  "version": "1.0",
  "createdAt": "2026-10-17T14:00:47.429Z",
  "authentication": {
    "type": "NO_AUTH"
  },
  "interaction": {
    "mode": "PUSH",
    "subscribe": {
      "protocol": "HTTP",
      "urlTemplate": "{{__SUB_URL__}}",
      "method": "POST",
      "headers": {},
      "bodyTemplate": "{\"url\": \"{{__SYS_OWN_URL_DOCKER__}}{{__RUNTIME_INGRESS__}}\", \"event_type\": \"hotel_waste_generated\"}"
    },
    "unsubscribe": {
      "protocol": "HTTP",
      "urlTemplate": "{{__UNSUB_URL__}}/{{__RUNTIME_JSON_{.subscription.data.ID}__}}",
      "method": "POST",
      "headers": {},
      "bodyTemplate": "{}"
    }
  },
  "parameters": [
    {
      "name": "SUB_URL",
      "title": "Subscription URL",
      "description": null,
      "paramType": "STRING",
      "required": true,
      "defaultVal

In [14]:
payload = {
    "templateName": template["name"],
    "templateVersion": template["version"],
    "distributionId": distribution_push_id,
    "parameters": {
        "SUB_URL": "http://127.0.0.1:8082/subscriptions/subscribe",
        "UNSUB_URL": "http://127.0.0.1:8082/subscriptions/unsubscribe"
    }
}
instance = show(requests.post(f"{data_space_provider}/api/v1/connector/instances", headers=json_header, json=payload))
push_connector = show(requests.get(f"{data_space_provider}/api/v1/connector/instances/distribution/{distribution_push_id}", headers=json_header))

HTTP 200 POST http://127.0.0.1:1200/api/v1/connector/instances
{
  "id": "urn:connector-instance:010ddecb-a835-446e-8d60-dfaf2a6057db",
  "userId": "system",
  "name": "urn:connector-template:cddb3021-091a-4e2a-94d7-9c1920abaa86",
  "author": null,
  "description": null,
  "version": "1.0",
  "createdAt": "2026-10-17T14:00:47.502208Z",
  "authenticationConfig": {
    "type": "NO_AUTH"
  },
  "interaction": {
    "mode": "PUSH",
    "subscribe": {
      "protocol": "HTTP",
      "urlTemplate": "http://127.0.0.1:8082/subscriptions/subscribe",
      "method": "POST",
      "headers": {},
      "bodyTemplate": "{\"url\": \"http://host.docker.internal:1200{{__RUNTIME_INGRESS__}}\", \"event_type\": \"hotel_waste_generated\"}"
    },
    "unsubscribe": {
      "protocol": "HTTP",
      "urlTemplate": "http://127.0.0.1:8082/subscriptions/unsubscribe/{{__RUNTIME_JSON_{.subscription.data.ID}__}}",
      "method": "POST",
      "headers": {},
      "bodyTemplate": "{}"
    }
  },
  "distributionI

## 8. Catalog and dataset discovery by the Consumer (DSP 6.2.1, DSP 6.2.2 & `/api/v1/catalog-agent/peer-catalogs`)

The Consumer asks its own agent to fetch the Provider's catalog over DSP
(`POST /dsp/current/catalog/rpc/setup-catalog-request`, sending `CatalogRequestMessage`) and can
also request a single dataset (`POST /dsp/current/catalog/rpc/setup-dataset-request`, sending
`DatasetRequestMessage`). When fetched with `noCache: false`, the catalog is also cached and
exposed under `GET /api/v1/catalog-agent/peer-catalogs/{peerId}`.

From the answer we keep:
- `target_id`: the `@id` of the dataset created above.
- `policy_id`: the `@id` of one of its offers. A negotiation must open on an offer the catalog
  publishes, on the dataset that publishes it.
- `data_space_provider_dsp_endpoint`: the `endpointURL` of the Provider's data service (`service`
  is a list).

In [15]:
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "filter": [],
    "noCache": True,
}
provider_catalog = show(requests.post(f"{data_space_consumer}/dsp/current/catalog/rpc/setup-catalog-request", headers=json_header, json=payload))["response"]

HTTP 200 POST http://127.0.0.1:1100/dsp/current/catalog/rpc/setup-catalog-request
{
  "request": {
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
    "filter": [],
    "noCache": true
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "Catalog",
    "@id": "urn:catalog:7d98ad53-dc19-4f07-aa14-e91292d25047",
    "identifier": "urn:catalog:7d98ad53-dc19-4f07-aa14-e91292d25047",
    "issued": "2026-10-16T21:16:00.204583",
    "participantId": "did

In [16]:
dataset_rpc = show(requests.post(
    f"{data_space_consumer}/dsp/current/catalog/rpc/setup-dataset-request",
    headers=json_header,
    json={
        "associatedAgentPeer": data_space_provider_participant_id,
        "dataset": dataset_id,
        "noCache": True,
    },
))["response"]

published = next(d for d in provider_catalog["dataset"] if d["@id"] == dataset_id)
assert dataset_rpc["@id"] == published["@id"]

target_id = published["@id"]
policy_id = published["hasPolicy"][0]["@id"]
data_space_provider_dsp_endpoint = provider_catalog["service"][0]["endpointURL"]

# Marker read by the guards of the negotiation and transfer notebooks
catalog_setup = "no_auth"

print(f"Dataset (target):      {target_id}")
print(f"Offer (policy):        {policy_id}")
print(f"Provider DSP address:  {data_space_provider_dsp_endpoint}")

HTTP 200 POST http://127.0.0.1:1100/dsp/current/catalog/rpc/setup-dataset-request
{
  "request": {
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
    "dataset": "urn:dataset:9b38dfda-bcb8-4968-a826-294c7905c929",
    "noCache": true
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "Dataset",
    "@id": "urn:dataset:9b38dfda-bcb8-4968-a826-294c7905c929",
    "identifier": "urn:dataset:9b38dfda-bcb8-4968-a826-294c7905c929",
    "issued": "2026-